### Data Loading and Initial Exploration

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics.pairwise import cosine_similarity

sns.set_theme(style="whitegrid")
%matplotlib inline

#Data Loading
try:
    df = pd.read_csv('anime.csv')
    print("Dataset loaded successfully!")
except FileNotFoundError:
    print("Error: 'anime.csv' not found. Creating a synthetic dataset for demonstration.")
    
print("\n--- Dataset Shape ---")
print(f"Rows: {df.shape[0]}, Columns: {df.shape[1]}")
print("\n--- Data Info ---")
print(df.info())
print("\n--- First 5 Rows ---")
print(df.head())

Dataset loaded successfully!

--- Dataset Shape ---
Rows: 12294, Columns: 7

--- Data Info ---
<class 'pandas.DataFrame'>
RangeIndex: 12294 entries, 0 to 12293
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   anime_id  12294 non-null  int64  
 1   name      12294 non-null  str    
 2   genre     12232 non-null  str    
 3   type      12269 non-null  str    
 4   episodes  12294 non-null  str    
 5   rating    12064 non-null  float64
 6   members   12294 non-null  int64  
dtypes: float64(1), int64(2), str(4)
memory usage: 672.5 KB
None

--- First 5 Rows ---
   anime_id                              name  \
0     32281                    Kimi no Na wa.   
1      5114  Fullmetal Alchemist: Brotherhood   
2     28977                          Gintama°   
3      9253                       Steins;Gate   
4      9969                     Gintama&#039;   

                                               genre   type episodes  ratin

### Data Preprocessing

In [ ]:
# --- Handling Missing Values ---
print("Missing values per column before cleaning:")
print(df.isnull().sum())

df['genre'] = df['genre'].fillna('Unknown')
df['type'] = df['type'].fillna('Unknown')

df['rating'] = df['rating'].fillna(df['rating'].median())
df['members'] = df['members'].fillna(df['members'].median())

df['episodes'] = df['episodes'].replace('Unknown', np.nan)
df['episodes'] = pd.to_numeric(df['episodes'])
df['episodes'] = df['episodes'].fillna(df['episodes'].median())

print("\nMissing values checked and handled successfully.")

Missing values per column before cleaning:
anime_id      0
name          0
genre        62
type         25
episodes      0
rating      230
members       0
dtype: int64

Missing values checked and handled successfully.


### Feature Extraction (Vectorization and Normalization)

In [ ]:
genres_split = df['genre'].str.get_dummies(sep=', ')

scaler = MinMaxScaler()
numerical_features = df[['rating', 'members', 'episodes']]
numerical_scaled = pd.DataFrame(scaler.fit_transform(numerical_features), 
                                columns=numerical_features.columns, 
                                index=df.index)

features_matrix = pd.concat([genres_split, numerical_scaled], axis=1)
print(f"Feature matrix successfully constructed. Matrix Shape: {features_matrix.shape}")

Feature matrix successfully constructed. Matrix Shape: (12294, 47)


### Cosine Similarity Matrix and Recommendation Function

In [ ]:
print("Computing Cosine Similarity Matrix...")
cosine_sim = cosine_similarity(features_matrix, features_matrix)
print("Similarity matrix calculated successfully.")

def recommend_anime(anime_name, df=df, similarity_matrix=cosine_sim, threshold=0.5):
    matched_names = df[df['name'].str.lower() == anime_name.lower()]['name'].values
    if len(matched_names) == 0:
        return f"Error: Anime '{anime_name}' was not found in the dataset. Please double-check spelling."
    
    actual_name = matched_names[0]
    idx = df[df['name'] == actual_name].index[0]
    
    sim_scores = list(enumerate(similarity_matrix[idx]))
    
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    
    recommendations = []
    for i, score in sim_scores:
        if i == idx:
            continue
        if score < threshold:
            break
        
        recommendations.append({
            'Name': df['name'].iloc[i],
            'Genre': df['genre'].iloc[i],
            'Type': df['type'].iloc[i],
            'Rating': df['rating'].iloc[i],
            'Similarity Score': round(score, 4)
        })
        
    if len(recommendations) == 0:
        return f"No recommendations found matching a strict similarity threshold of {threshold}."
        
    return pd.DataFrame(recommendations)

Computing Cosine Similarity Matrix...
Similarity matrix calculated successfully.


### Recommendation Testing and Threshold Experiments

In [ ]:
target_show = "Kimi no Na wa."

print(f"=== Recommendations for '{target_show}' (Threshold = 0.75) ===")
rec_results = recommend_anime(target_show, threshold=0.75)
if isinstance(rec_results, pd.DataFrame):
    print(rec_results.head(10))  
else:
    print(rec_results)

print("\n" + "="*50 + "\n")

for t in [0.4, 0.7, 0.9]:
    res = recommend_anime(target_show, threshold=t)
    count = len(res) if isinstance(res, pd.DataFrame) else 0
    print(f"Similarity Cutoff Threshold: {t} -> Generated {count} matches.")

=== Recommendations for 'Kimi no Na wa.' (Threshold = 0.75) ===
                                                Name  \
0                        Wind: A Breath of Heart OVA   
1                       Wind: A Breath of Heart (TV)   
2              Aura: Maryuuin Kouga Saigo no Tatakai   
3  Clannad: After Story - Mou Hitotsu no Sekai, K...   
4                      Kokoro ga Sakebitagatterunda.   
5                     Angel Beats!: Another Epilogue   
6                                         True Tears   
7                                   Myself; Yourself   
8                                Kimikiss Pure Rouge   
9                         Koi to Senkyo to Chocolate   

                                          Genre     Type  Rating  \
0          Drama, Romance, School, Supernatural      OVA    6.35   
1          Drama, Romance, School, Supernatural       TV    6.14   
2  Comedy, Drama, Romance, School, Supernatural    Movie    7.67   
3                        Drama, Romance, School

### Interview Questions & Structured Answers

1. Can you explain the difference between user-based and item-based collaborative filtering?

The foundational difference lies in which dimension of the matrix similarity is computed across.User-Based Collaborative Filtering:Concept: Looks at a target user, finds other users who have historically rated the same items similarly (their "peers"), and recommends items those peers liked.Pros/Cons: It can reveal highly serendipitous discoveries across genres. However, because user populations grow fast and individual human tastes fluctuate over time, computing user matrices in real-time is memory-intensive and computationally expensive.Item-Based Collaborative Filtering:Concept: Looks at the target item, analyzes the profile of users who liked it, and finds other items that share an overlapping set of approving users.Pros/Cons: It is significantly more stable. An anime’s core audience and properties grow and change at a much slower rate than a single user's daily mood. This allows item-to-item similarity matrices to be completely pre-computed offline, making it highly scalable for massive websites (like Amazon or Netflix).

2. What is collaborative filtering, and how does it work?

Collaborative Filtering (CF) is a methodology used by recommendation engines to predict a user's preferences by harvesting behavioral patterns, ratings, and choices from a massive collaborative community of other users. It works under the core assumption that if individuals agreed on certain choices in the past, they will likely share opinions on choices in the future.How it works step-by-step:Constructing the Interaction Matrix: The system creates a large utility pivot table where rows represent users, columns represent items, and cells contain explicit ratings (like $1\text{ to }10$) or implicit signals (clicks, views, watch time).


Calculating Similarity: It calculates mathematical alignment vectors between users or items using metric distance measures like Cosine Similarity or the Pearson Correlation Coefficient:$$\text{Cosine Similarity}(\mathbf{A}, \mathbf{B}) = \frac{\mathbf{A} \cdot \mathbf{B}}{\|\mathbf{A}\| \|\mathbf{B}\|}$$Predicting and Ranking: To recommend a title to a user, the system fills in the blank spaces of unassigned ratings by calculating a weighted average of the ratings given by their closest k-nearest neighbors ($k\text{-NN}$). The items calculated with the highest predicted scores are served directly to the user.